# IDS 706: Rust exercises

Based on [Kedar-V's course notebook](https://github.com/Kedar-V/data-processing-frameworks-demo/blob/75772a44ed1cbc61e66396fad49a9aa0c913c8ef/notebooks/rust_vs_python_intro.ipynb). I kept the small movie-rating examples and added changes to try out mutability, ownership, and borrowing.

Run this with the **Rust** kernel. Three cells are supposed to give compiler errors; the working examples follow them. The outputs below are saved from running the notebook.


## Basic syntax

These first exercises cover functions, conditions, and loops. They use the examples from class rather than the Treasury dataset.


### Hello, Rust

In a normal Rust program, execution starts at `main`. In this notebook, I call `main()` explicitly.


In [1]:
// Hello-world example from class.
fn main() {
    let engine = "Rust";
    println!("Hello from {engine}.");
}

main(); // Call main explicitly in the notebook.


Hello from Rust.


The cell prints `Hello from Rust.`. The later cells can run statements directly without putting each example in `main`.


For the review exercise, I filled in my name and used a sample movie title and rating.


In [2]:
// Sample values for this introductory exercise.
fn main() {
    let name = "Haoran";
    let movie = "Example movie";
    let stars = 4;
    println!("{name}'s review");
    println!("  {movie}");
    println!("  {stars} / 5 stars");
}
main();


Haoran's review


  Example movie


  4 / 5 stars


### Conditions

The branches work like Python, but Rust uses braces and `else if` instead of `elif`.


In [3]:
let rating = 4.5;

if rating >= 4.0 {
    println!("{rating} stars: high");
} else if rating >= 3.0 {
    println!("{rating} stars: in the middle");
} else {
    println!("{rating} stars: low");
}


4.5 stars: high


()

For the cutoff exercise, I tried 3212 and 500. A count of at least 1000 should print `keep`; a smaller count should print `skip`.


In [4]:
// Check both sides of the cutoff.
let n = 3212;
if n >= 1000 {
    println!("{n}: keep");
} else {
    println!("{n}: skip");
}
let n = 500;
if n >= 1000 {
    println!("{n}: keep");
} else {
    println!("{n}: skip");
}


3212: keep


500: skip


()

### Loops

`0..3` gives 0, 1, and 2. The next loop prints each rating in the array.


In [5]:
for n in 0..3 { // 0, 1, 2
    println!("pass {n}");
}

let ratings = [4.5, 3.0, 5.0];
for rating in ratings {
    println!("rating = {rating:.1}");
}


pass 0


pass 1


pass 2


rating = 4.5


rating = 3.0


rating = 5.0


()

I used a counter and an `if` statement to count ratings of 4.0 or higher. The result is 3.


In [6]:
let ratings = [4.5, 3.0, 5.0, 2.0, 5.0];
let mut high = 0;
for rating in ratings {
    if rating >= 4.0 {
        high += 1;
    }
}
println!("high ratings: {high}");


high ratings: 3


## Mutability

A variable declared with `let` cannot be reassigned unless it has `mut`. In this example, `rows_read` needs to change inside the loop, but the cutoff stays fixed.


In [7]:
// Fixed cutoff.
let min_ratings = 1000;
println!("keep movies with at least {min_ratings} ratings");

// This count changes in the loop.
let mut rows_read = 0;
for _ in 0..3 { // `_` means we do not use the loop counter
    rows_read += 100_000;
    println!("rows_read = {rows_read}");
}


keep movies with at least 1000 ratings


rows_read = 100000


rows_read = 200000


rows_read = 300000


()

Here I try changing the cutoff without `mut`. This cell is expected to fail.


In [8]:
let min_ratings = 1000;
println!("keep movies with at least {min_ratings} ratings");

min_ratings = 2000;
println!("keep movies with at least {min_ratings} ratings");


Error: cannot assign twice to immutable variable `min_ratings`

Rust reports E0384 because `min_ratings` is immutable. The next cell adds `mut` to the declaration.


In [9]:
// Same example, now with mut.
let mut min_ratings = 1000;
println!("before: {min_ratings}");
min_ratings = 2000;
println!("after: {min_ratings}");


before: 1000


after: 2000


With `mut`, the output changes from `before: 1000` to `after: 2000`.


Adding `mut` makes reassignment possible. The next examples check what happens when a vector is assigned to another variable.


## Ownership and cloning

For a Python list, `b = a` gives another name for the same list. Assigning a Rust vector to a new variable moves ownership instead.


In [10]:
// A vector of sample ratings.
let ratings = vec![4.5, 3.0, 5.0];
println!("ratings = {ratings:?}"); // `:?` prints the list in a readable way


ratings = [4.5, 3.0, 5.0]


This cell moves the vector to `moved`, then tries to print it using both names.


In [11]:
let ratings = vec![4.5, 3.0, 5.0];
let moved = ratings;
println!("{ratings:?} {moved:?}");


Error: borrow of moved value: `ratings`

The result is E0382: `ratings` cannot be used after the move. The next example makes a separate copy with `clone()` before moving the original.


In [12]:
let ratings = vec![4.5, 3.0, 5.0];
let copy = ratings.clone(); // Separate copy.
let moved = ratings; // The original vector moves here.
println!("moved = {moved:?}");
println!("copy  = {copy:?}");


moved = [4.5, 3.0, 5.0]


copy  = [4.5, 3.0, 5.0]


To check whether the clone is separate, I added 1.0 to `copy` and printed both vectors.


In [13]:
// Add to the copy and compare it with the original.
let ratings = vec![4.5, 3.0, 5.0];
let mut copy = ratings.clone();
copy.push(1.0);
println!("ratings = {ratings:?}");
println!("copy    = {copy:?}");


ratings = [4.5, 3.0, 5.0]


copy    = [4.5, 3.0, 5.0, 1.0]


`copy` has four values after the change, while `ratings` still has three. Adding to the cloned vector did not change the original.


## Borrowing

I used two shared references to read the same vector. After those reads, I used one mutable reference to add a value.


In [14]:
let mut ratings = vec![5, 3, 4, 4];
{
    let first = &ratings;
    let second = &ratings;
    println!("two readers: {first:?} and {second:?}");
}
{
    // The shared references are no longer used here.
    let writer = &mut ratings;
    writer.push(2);
}
println!("after a write: {ratings:?}");


two readers: [5, 3, 4, 4] and [5, 3, 4, 4]


after a write: [5, 3, 4, 4, 2]


Both shared references printed the same values. The mutable reference then added 2, giving `[5, 3, 4, 4, 2]`. `ratings` still owns the vector.


### Changing a vector while reading it

I changed the original loop from `for rating in ratings` to `for rating in &ratings`, with `*rating` in the condition. This tests a borrowing conflict rather than moving the vector.

The loop reads through a shared reference and tries to remove an item from that same vector. This cell should fail.


In [15]:
// Expected error: changing ratings while the loop borrows it.
let mut ratings = vec![1, 2, 2, 3, 4, 5];
for rating in &ratings {
    if *rating == 2 {
        ratings.remove(1);
    }
}
println!("{ratings:?}");


Error: cannot borrow `ratings` as mutable because it is also borrowed as immutable

Rust reports E0502: the vector cannot be borrowed mutably while the loop is still borrowing it immutably. The call to `remove` conflicts with the loop reading `ratings`.


For a working version, I read from `ratings` and put the values other than 2 into a separate vector, `kept`.


In [16]:
let ratings = vec![1, 2, 2, 3, 4, 5];
let mut kept = Vec::new();
for rating in &ratings {
    if *rating != 2 {
        kept.push(*rating);
    }
}
println!("original: {ratings:?}");
println!("filtered: {kept:?}");


original: [1, 2, 2, 3, 4, 5]


filtered: [1, 3, 4, 5]


The result is `[1, 3, 4, 5]`, so both 2s are gone. The original vector is unchanged.


## Notes

The main difference from Python in these examples is that Rust checks ownership and borrowing before running the code. I needed `mut` for changes, `clone()` for a separate vector, and non-conflicting borrows when reading or writing.


Polars connects this to the Python part of the assignment: its engine uses Rust, while the analysis script calls it from Python.


## Optional: scope and memory

I changed the buffer size from 64 MB to 16 MB. This example prints a message when the buffer is created and another when it is dropped.


In [17]:
// Print when this buffer is dropped.
struct Buffer {
    data: Vec<u8>, // the actual bytes
}

fn make_buffer(megabytes: usize) -> Buffer {
    println!("  allocate {megabytes} MB");
    Buffer {
        data: vec![0; megabytes * 1024 * 1024],
    }
}

// Runs when the buffer goes out of scope.
impl Drop for Buffer {
    fn drop(&mut self) {
        let megabytes = self.data.len() / 1024 / 1024;
        println!("  FREE {megabytes} MB");
    }
}

println!("enter outer block");
{
    println!("  enter inner scope");
    let _scratch = make_buffer(16); // Use 16 MB for this example.
    println!("  inner scope is about to end");
} // The buffer is dropped here.
println!("back in the outer block: that memory is already gone");


enter outer block


  enter inner scope


  allocate 16 MB


  inner scope is about to end


  FREE 16 MB


back in the outer block: that memory is already gone


The output prints `FREE 16 MB` before `back in the outer block`. The buffer is dropped when the inner scope ends.


There is no manual `free()` call here. Leaving the scope runs `Drop` for the buffer.
